# 01 — EDA
Exploratory analysis for PD credit scoring: Lending Club data.

**Key sections:**
1. Data overview & distributions
2. Target variable analysis
3. **Temporal distribution analysis** — statistical proof of regime changes
4. Feature correlations
5. Train/val/test split justification

## 1. Setup & Load Data

In [ ]:
%pip install -q numpy pandas matplotlib seaborn scipy

import sys
from pathlib import Path

ROOT = Path("..").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("ROOT =", ROOT)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid", palette="muted")

# Load data — prefer Lending Club, fallback to sample
DATA = ROOT / "data"
lc_path = DATA / "lending_club.csv"
sample_path = DATA / "sample_applications.csv"

if lc_path.exists():
    df = pd.read_csv(lc_path, nrows=500_000)  # sample for EDA speed
    print(f"Loaded Lending Club: {df.shape}")
    # Parse dates
    if "issue_d" in df.columns:
        df["application_date"] = pd.to_datetime(df["issue_d"], format="%b-%Y", errors="coerce")
    # Target
    if "loan_status" in df.columns:
        defaults = {"Charged Off", "Default", "Late (31-120 days)",
                    "Does not meet the credit policy. Status:Charged Off"}
        df["is_default"] = df["loan_status"].isin(defaults).astype(int)
elif sample_path.exists():
    df = pd.read_csv(sample_path)
    print(f"Loaded sample: {df.shape}")
    if "issue_d" in df.columns:
        df["application_date"] = pd.to_datetime(df["issue_d"], format="%b-%Y", errors="coerce")
    if "loan_status" in df.columns:
        defaults = {"Charged Off", "Default", "Late (31-120 days)"}
        df["is_default"] = df["loan_status"].isin(defaults).astype(int)
else:
    from scripts.generate_sample_data import generate
    DATA.mkdir(exist_ok=True)
    generate(5000).to_csv(sample, index=False)
    print("generated", sample)
df = pd.read_csv(sample)
df.head()

In [ ]:
# KS test: are key feature distributions different between eras?
print("=" * 70)
print("KOLMOGOROV-SMIRNOV TEST: Feature distribution comparison")
print("=" * 70)

features_to_test = [c for c in ["fico_range_low", "annual_inc", "dti", "int_rate", "loan_amnt"] if c in df.columns]
era_names = list(era_dfs.keys())

for feat in features_to_test:
    print(f"\n--- {feat} ---")
    for i in range(len(era_names)):
        for j in range(i + 1, len(era_names)):
            a = era_dfs[era_names[i]][feat].dropna()
            b = era_dfs[era_names[j]][feat].dropna()
            if len(a) > 10 and len(b) > 10:
                ks_stat, ks_pval = stats.ks_2samp(a, b)
                sig = "***" if ks_pval < 0.001 else "**" if ks_pval < 0.01 else "*" if ks_pval < 0.05 else "ns"
                print(f"  {era_names[i]} vs {era_names[j]}: KS={ks_stat:.4f}, p={ks_pval:.2e} {sig}")

In [ ]:
if "loan_status" in df.columns:
    defaults = {"Charged Off", "Default", "Late (31-120 days)"}
    df["is_default"] = df["loan_status"].isin(defaults).astype(int)
    print("default rate", df["is_default"].mean())
    df["is_default"].value_counts(normalize=True).plot(kind="bar", title="Default rate")
    plt.show()

In [ ]:
cols = [c for c in ["loan_amnt", "annual_inc", "dti", "int_rate", "fico_range_low"] if c in df.columns]
if cols:
    df[cols].hist(bins=30, figsize=(12, 8))
    plt.tight_layout()
    plt.show()

In [ ]:
if "issue_d" in df.columns:
    dates = pd.to_datetime(df["issue_d"], format="%b-%Y", errors="coerce")
    print(dates.min(), dates.max())
    dates.dt.to_period("M").value_counts().sort_index().plot(figsize=(12, 3), title="Applications per month")
    plt.show()